In [ ]:
!whoami
!python -V

root
Python 3.12.12


In [ ]:
!pip install evaluate
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps

In [ ]:
# Load the training data from a JSON Lines file (one JSON object per line)
# train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
# train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
# kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
# kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

# Load the training data line by line to handle potential parsing issues
train_data_list = []
with open('sample_data/train.jsonl', 'r') as f:
#with open('train.jsonl', 'r') as f:
    for line in f:
        try:
            train_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()}")
            print(f"Error message: {e}")

train_data = pd.DataFrame(train_data_list)
train_data = json_normalize(train_data.to_dict(orient='records'))


# Load the Kaggle test data line by line
kaggle_data_list = []
with open('sample_data/kaggle_test.jsonl', 'r') as f:
    for line in f:
        try:
            kaggle_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()}")
            print(f"Error message: {e}")

kaggle_data = pd.DataFrame(kaggle_data_list)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

Error decoding JSON on line: {"quoted_status": {"extended_tweet": {"entities": {"urls": [], "hashtags": [{"indices": [13, 25], "text": "IleDeFrance"}, {"indices": [42, 54], "text": "confinement"}, {"indices": [161, 169], "text": "COVID19"}, {"indices": [170, 184], "text": "COVID19France"}], "user_mentions": [], "symbols": []}, "full_text": "\ud83c\udde8\ud83c\uddf5 FLASH - L'#IleDeFrance ne vivra pas un #confinement le week-end. La piste a \u00e9t\u00e9 \u00e9cart\u00e9e par le gouvernement lors du conseil de d\u00e9fense ce matin. (FranceInfo) #COVID19 #COVID19France", "display_text_range": [0, 184]}, "in_reply_to_status_id_str": null, "in_reply_to_status_id": null, "created_at": "Wed Mar 03 18:40:00 +0000 2021", "in_reply_to_user_id_str": null, "source": "<a href=\"https://about.twitter.com/products/tweetdeck\" rel=\"nofollow\">TweetDeck</a>", "retweet_count": 972, "retweeted": false, "geo": null, "filter_level": "low", "in_reply_to_screen_name": null, "is_quote_status": false, "id_s

In [ ]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train['full_text'] = X_train.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle['full_text'] = X_kaggle.apply(lambda tweet: extract_full_text(tweet), axis=1)

In [ ]:
# ================================================
# 1. LOAD + CLEAN JSONL DATA
# ================================================
import json
import pandas as pd
from pandas import json_normalize

# ---------------------- TRAIN ----------------------
train_data_list = []
with open("sample_data/train.jsonl", "r") as f:
    for line in f:
        try:
            train_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("JSON error:", e)

train_data = json_normalize(train_data_list)

# ---------------------- KAGGLE TEST ----------------------
kaggle_data_list = []
with open("sample_data/kaggle_test.jsonl", "r") as f:
    for line in f:
        try:
            kaggle_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("JSON error:", e)

kaggle_data = json_normalize(kaggle_data_list)


# ================================================
# 2. EXTRACT FULL TEXT (EXTENDED TWEETS)
# ================================================
def extract_full_text(row):
    if "extended_tweet.full_text" in row and not pd.isna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    return row.get("text", "")

train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)


# ================================================
# 3. BUILD HUGGINGFACE DATASETS
# ================================================
from datasets import Dataset

raw_train_dataset = Dataset.from_pandas(train_data[["full_text", "label"]])
raw_kaggle_dataset = Dataset.from_pandas(kaggle_data[["full_text"]])

# Train/validation split
raw_split = raw_train_dataset.train_test_split(test_size=0.1)
raw_train_dataset = raw_split["train"]
raw_val_dataset = raw_split["test"]


# ================================================
# 4. TOKENIZATION
# ================================================
from transformers import AutoTokenizer

model_name = "nreimers/MiniLM-L6-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["full_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

ds_train = raw_train_dataset.map(tokenize_fn, batched=True, load_from_cache_file=False)
ds_val   = raw_val_dataset.map(tokenize_fn, batched=True, load_from_cache_file=False)

# HF Trainer expects "labels"
ds_train = ds_train.rename_column("label", "labels")

ds_train.set_format("torch")
ds_val.set_format("torch")


# ================================================
# 5. TRAIN MINILM
# ================================================
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="minilm_final",
    report_to="none",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs_minilm",
    fp16=True,
    save_total_limit=1,
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
)

print("🚀 Starting MiniLM training...")
trainer.train()
print("✅ Training finished.")


# ================================================
# 6. SAVE THE TRAINED MODEL
# ================================================
trainer.save_model("minilm_final")
tokenizer.save_pretrained("minilm_final")
print("📦 Model saved in minilm_final/")


# ================================================
# 7. GENERATE KAGGLE PREDICTIONS
# ================================================
# Tokenize Kaggle test set
ds_kaggle = raw_kaggle_dataset.map(
    tokenize_fn,
    batched=True,
    load_from_cache_file=False
)
ds_kaggle.set_format("torch")

# Predict
preds = trainer.predict(ds_kaggle).predictions
pred_labels = preds.argmax(axis=1)

# Build submission CSV
submission = pd.DataFrame({
    "id": kaggle_data["id"],
    "label": pred_labels
})

submission.to_csv("submission.csv", index=False)
print("📄 Saved submission.csv")



JSON error: Unterminated string starting at: line 1 column 5446 (char 5445)
JSON error: Expecting ',' delimiter: line 1 column 2299 (char 2298)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1286930571.py:106: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting MiniLM training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [ ]:
# -------------------------------------------------------------
# Initialisation du modèle
# -------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# -------------------------------------------------------------
# Arguments d'entraînement simplifiés
# -------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="alberta-v2",
    report_to="none",   # <-- disables wandb
    learning_rate=5e-5,      # plus sûr pour DeBERTa
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs_final",
    fp16=True,               # activer FP16
    save_total_limit=1,
    gradient_checkpointing=True, # Enable gradient checkpointing directly here
)

# model.gradient_checkpointing_enable()  # Removed as it's now handled by TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -------------------------------------------------------------
# Lancer l'entraînement
# -------------------------------------------------------------
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete.")

# -------------------------------------------------------------
# Sauvegarde du modèle + tokenizer
# -------------------------------------------------------------
trainer.save_model("deberta_small_final")
tokenizer.save_pretrained("deberta_small_final")

print("📦 Modèle sauvegardé dans 'deberta_small_final'")

In [ ]:
from torch.utils.data import Subset
import random

# Choisir un échantillon aléatoire
sample_size = 1000
indices = random.sample(range(len(ds_train)), sample_size)
train_sample = Subset(ds_train, indices)

# Évaluer le modèle
metrics = trainer.evaluate(eval_dataset=train_sample)
print("Accuracy sur l'échantillon random du jeu d'entraînement :", metrics["eval_accuracy"])


In [ ]:
# -------------------------------------------------------------
# Évaluation sur le test set (Kaggle)
# -------------------------------------------------------------
from datasets import Dataset
import pandas as pd
import numpy as np

# Keep all columns so we can use challenge_id later
X_kaggle_ds = Dataset.from_pandas(X_kaggle)

# Tokenize the text
X_kaggle_ds = X_kaggle_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=[col for col in X_kaggle_ds.column_names if col != "challenge_id"]  # keep challenge_id
)

# Make predictions
preds = trainer.predict(X_kaggle_ds).predictions
y_pred_test = np.argmax(preds, axis=1)

# -------------------------------------------------------------
# Sauvegarde des prédictions
# -------------------------------------------------------------
output = pd.DataFrame({
    'ID': X_kaggle_ds['challenge_id'],
    'Prediction': y_pred_test
})
output.to_csv('deberta_v3_small_predictions.csv', index=False)

print("\n✅ Predictions saved to 'deberta_v3_small_predictions.csv'")


START OF MODEL_CHECKPOINT_PLANT.PY just below :

In [5]:
# ================================================
# 0. IMPORTS
# ================================================
import json
import pandas as pd
from pandas import json_normalize
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, f1_score
import numpy as np


# ================================================
# 1. LOAD + CLEAN JSONL DATA
# ================================================
def load_jsonl(path):
    data_list = []
    with open(path, "r") as f:
        for line in f:
            try:
                data_list.append(json.loads(line))
            except json.JSONDecodeError as e:
                print("JSON error:", e)
    return json_normalize(data_list)


train_data = load_jsonl("train.jsonl")
kaggle_data = load_jsonl("kaggle_test.jsonl")


# ================================================
# 2. EXTRACT FULL TEXT (EXTENDED TWEETS)
# ================================================
def extract_full_text(row):
    if "extended_tweet.full_text" in row and not pd.isna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    return row.get("text", "")

train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)


# ============================================================
# 3. BUILD HUGGINGFACE DATASETS
# ============================================================
raw_train_dataset = Dataset.from_pandas(train_data[["full_text", "label"]])
raw_kaggle_dataset = Dataset.from_pandas(kaggle_data[["full_text"]])

# Convert integer labels → ClassLabel (for stratification)
from datasets import ClassLabel
class_label = ClassLabel(num_classes=2, names=["observer", "influencer"])
raw_train_dataset = raw_train_dataset.cast_column("label", class_label)

# Stratified train/val split
raw_split = raw_train_dataset.train_test_split(
    test_size=0.1,
    stratify_by_column="label"
)

raw_train_dataset = raw_split["train"]
raw_val_dataset   = raw_split["test"]


# ================================================
# 4. TOKENIZATION
# ================================================
model_name = "nreimers/MiniLM-L6-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["full_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

ds_train = raw_train_dataset.map(tokenize_fn, batched=True, load_from_cache_file=False)
ds_val   = raw_val_dataset.map(tokenize_fn, batched=True, load_from_cache_file=False)

# Trainer expects "labels"
ds_train = ds_train.rename_column("label", "labels")

ds_train.set_format("torch")
ds_val.set_format("torch")


# ================================================
# 5. METRICS
# ================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


# ================================================
# 6. TRAIN MINILM (COLAB GPU-OPTIMIZED)
# ================================================
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="minilm_final",
    report_to="none",
    learning_rate=3e-5,
    per_device_train_batch_size=32,   # larger batch fits on GPU
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs_minilm",

    # 🚀 GPU acceleration
    fp16=True,          # fastest on T4/V100/A100
    bf16=False,         # NVIDIA fp16 preferred
    no_cuda=False,      # ensures GPU is used
    torch_compile=True, # PyTorch 2.0 speed boost

    save_total_limit=1,
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("🚀 Starting MiniLM training on GPU...")
trainer.train()
print("✅ Training finished.")


# ================================================
# 7. SAVE TRAINED MODEL
# ================================================
trainer.save_model("minilm_final")
tokenizer.save_pretrained("minilm_final")
print("📦 Model saved in minilm_final/")


# ================================================
# 8. GENERATE KAGGLE PREDICTIONS
# ================================================
ds_kaggle = raw_kaggle_dataset.map(
    tokenize_fn,
    batched=True,
    load_from_cache_file=False
)
ds_kaggle.set_format("torch")

preds = trainer.predict(ds_kaggle).predictions
pred_labels = preds.argmax(axis=1)

submission = pd.DataFrame({
    "id": kaggle_data["id"],
    "label": pred_labels
})

submission.to_csv("submission.csv", index=False)
print("📄 Saved submission.csv")


Casting the dataset:   0%|          | 0/154914 [00:00<?, ? examples/s]

Map:   0%|          | 0/139422 [00:00<?, ? examples/s]

Map:   0%|          | 0/15492 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The speedups for torchdynamo mostly come with GPU Ampere or higher and which is not detected here.
/tmp/ipython-input-2374112392.py:133: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting MiniLM training on GPU...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
W1123 14:09:40.377000 165 torch/utils/cpp_extension.py:117] [0/0] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [6]:
!pip install -q "transformers>=4.35.0" datasets accelerate peft scikit-learn bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.2 MB/s eta 0:00:00


In [3]:
# ================================================
# 0. IMPORTS & GPU CHECK
# ================================================
import os
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, f1_score
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    raise SystemError("GPU NOT detected. In Colab, go to Runtime → Change Runtime Type → GPU.")

device = torch.device("cuda")


# ================================================
# 1. LOAD JSONL DATA
# ================================================
def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except:
                pass
    return json_normalize(rows)

train_data = load_jsonl("train.jsonl")
kaggle_data = load_jsonl("kaggle_test.jsonl")

print("Train rows:", len(train_data))
print("Kaggle rows:", len(kaggle_data))


# ================================================
# 2. EXTRACT FULL TEXT
# ================================================
def extract_full_text(row):
    if "extended_tweet.full_text" in row and not pd.isna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    return row.get("text", "")

train_data["full_text"]  = train_data.apply(extract_full_text, axis=1)
kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)


# ================================================
# 3. BUILD HF DATASETS
# ================================================
raw_train = Dataset.from_pandas(train_data[["full_text", "label"]])
raw_test  = Dataset.from_pandas(kaggle_data[["full_text"]])

from datasets import ClassLabel
labels = ClassLabel(num_classes=2, names=["observer", "influencer"])
raw_train = raw_train.cast_column("label", labels)

split = raw_train.train_test_split(test_size=0.1, stratify_by_column="label")
ds_train = split["train"]
ds_val   = split["test"]

print("Train:", len(ds_train), " Val:", len(ds_val))


# ================================================
# 4. TOKENIZATION (Dynamic padding)
# ================================================
MODEL_NAME = "nreimers/MiniLM-L6-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["full_text"], truncation=True, max_length=128)

ds_train = ds_train.map(tokenize, batched=True)
ds_val   = ds_val.map(tokenize, batched=True)
ds_test  = raw_test.map(tokenize, batched=True)

ds_train = ds_train.rename_column("label", "labels")
ds_train.set_format("torch")
ds_val.set_format("torch")
ds_test.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ================================================
# 5. METRICS
# ================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


# ================================================
# 6. LOAD BASE MODEL + LoRA
# ================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# LoRA config optimized for MiniLM
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "key", "value", "dense"],  # works for MiniLM
    lora_dropout=0.05,
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# ================================================
# 7. TRAINING ARGS (GPU + FP16 + EARLY STOPPING)
# ================================================
training_args = TrainingArguments(
    output_dir="lora_minilm_out",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,                     # early stopping will stop earlier
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,                              # use GPU fast path
    dataloader_num_workers=4,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=2,          # stops if 2 consecutive epochs have no improvement
        early_stopping_threshold=0.0001
    )]
)


# ================================================
# 8. TRAIN
# ================================================
print("🚀 Training with LoRA on GPU...")
trainer.train()
print("✅ Training complete!")


# ================================================
# 9. SAVE PEFT ADAPTER
# ================================================
model.save_pretrained("minilm_lora_adapter")
tokenizer.save_pretrained("minilm_lora_adapter")
print("📦 Saved LoRA adapter → minilm_lora_adapter/")


# ================================================
# 10. PREDICT TEST SET
# ================================================
preds = trainer.predict(ds_test).predictions
labels = preds.argmax(axis=1)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": labels
})
submission.to_csv("submission.csv", index=False)

print("📄 Saved submission.csv")


CUDA available: True
GPU device: Tesla T4


KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 160, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


KeyboardInterrupt: 

In [4]:
# ============================================================
# LOAD MODEL + TOKENIZER (BASE + LoRA ADAPTER)
# ============================================================
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from pandas import json_normalize
from tqdm.notebook import tqdm

BASE_MODEL = "nreimers/MiniLM-L6-H384-uncased"
ADAPTER_PATH = "minilm_lora_adapter"

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to("cuda")
model.eval()

print("✓ Model loaded on", next(model.parameters()).device)


# ============================================================
# LOAD KAGGLE TEST DATA
# ============================================================
def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except:
                pass
    return json_normalize(rows)

kaggle_data = load_jsonl("kaggle_test.jsonl")
print("Loaded", len(kaggle_data), "test samples")

def extract_full_text(row):
    if "extended_tweet.full_text" in row and not pd.isna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    return row.get("text", "")

kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)


# ============================================================
# PREDICT
# ============================================================
all_preds = []

print("Running predictions...")
for text in tqdm(kaggle_data["full_text"].tolist()):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=False
    ).to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=-1).item()

    all_preds.append(pred)


# ============================================================
# SAVE SUBMISSION CSV
# ============================================================
submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": all_preds
})

submission.to_csv("submission.csv", index=False)

print("📄 Saved submission.csv with", len(submission), "rows")
print(submission.head())



Loading tokenizer and base model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading LoRA adapter...
✓ Model loaded on cuda:0
Loaded 103380 test samples
Running predictions...


  0%|          | 0/103380 [00:00<?, ?it/s]

📄 Saved submission.csv with 103380 rows
   ID  Prediction
0   0           1
1   2           1
2   4           0
3   8           1
4   9           0


In [6]:
!zip -r /content/lora_minilm_out.zip /content/lora_minilm_out/
!zip -r /content/minilm_lora_adapter.zip /content/minilm_lora_adapter

from google.colab import files
files.download("/content/lora_minilm_out.zip")
files.download("/content/minilm_lora_adapter.zip")
files.download("/content/submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>